In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

# Klasifikacija antimikrobnih peptida

## Priprema podataka

Antimikrobni peptidi (AMP) su kratki lanci aminokiselina koji mogu djelovati protiv bakterija, gljivica i drugih mikroorganizama. Cilj ovog projekta je razviti model strojnog učenja koji može razlikovati antimikrobne peptide od peptida koji nemaju antimikrobno djelovanje.

U ovom dijelu projekta učitavaju se podaci, provjerava kvaliteta sekvenci te se priprema skup podataka koji će se koristiti u daljnjoj analizi.


In [ ]:
PROJECT_DIR = Path(".")

DATA_DIR = PROJECT_DIR / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Projektni direktorij:", PROJECT_DIR.resolve())
print("Raw direktorij:", RAW_DIR.resolve())
print("Processed direktorij:", PROCESSED_DIR.resolve())

Projektni direktorij: /content
Raw direktorij: /content/data/raw
Processed direktorij: /content/data/processed


## Čišćenje i priprema podataka

Prije izrade modela potrebno je provjeriti jesu li sekvence ispravne. Uklanjaju se neispravni zapisi i duplikati kako bi podaci bili što kvalitetniji. Nakon toga formira se skup podataka koji će se koristiti za treniranje modela.

In [ ]:
AA = set("ACDEFGHIKLMNPQRSTVWY")

def clean_sequence(seq):

    if pd.isna(seq):
        return None

    seq = str(seq).upper().strip()

    if len(seq) == 0:
        return None

    if not set(seq).issubset(AA):
        return None

    return seq

In [ ]:
dataset = pd.read_csv("peptides.csv")

print("Dimenzije:", dataset.shape)

print("\nStupci:")
print(dataset.columns.tolist())

dataset.head()

Dimenzije: (2000, 9)

Stupci:
['ID', 'COMPLEXITY', 'NAME', 'N TERMINUS', 'SEQUENCE', 'C TERMINUS', 'SYNTHESIS TYPE', 'TARGET GROUP', 'TARGET OBJECT']


,ID,COMPLEXITY,NAME,N TERMINUS,SEQUENCE,C TERMINUS,SYNTHESIS TYPE,TARGET GROUP,TARGET OBJECT
0,1,Multimer,Distinctin,NaN,NLVSGLIEARKYLEQLHRKLKNCKV ENREVPPGFTALIKTLR...,NaN,Ribosomal,Gram+ Gram-,Lipid Bilayer
1,3,Multimer,Halocidin,NaN,WLNALLHHGLNCAKGVLA ALLHHGLNCAKGVLA,AMD AMD,Ribosomal,Gram+,Lipid Bilayer
2,4,Multimer,Khal,NaN,KWLNALLHHGLNCAKGVLA ALLHHGLNCAKGVLA,AMD AMD,Synthetic,Gram+ Gram-,Lipid Bilayer
3,6,Multi-Peptide,Enterocin X,NaN,SNDSLWYGVGQFMGKQANCITNHPVKHMIIPGYCLSKILG IA...,NaN,Ribosomal,Gram+,Lipid Bilayer
4,7,Multi-Peptide,EAFP1 + EAFP2,NaN,XTCASRCPRPCNAGLCCSIYGYCGSGNAYCGAGNCRCQCRG X...,NaN,Ribosomal,Fungus,Lipid Bilayer


In [ ]:
import random


amp_df = dataset.copy()

amp_df["SEQUENCE"] = amp_df["SEQUENCE"].apply(clean_sequence)

amp_df = amp_df.dropna(subset=["SEQUENCE"])

amp_df = amp_df.drop_duplicates(subset=["SEQUENCE"])

amp_df = amp_df.reset_index(drop=True)

amp_df["length"] = amp_df["SEQUENCE"].str.len()

print("Broj AMP peptida:", len(amp_df))

amp_df["label"] = 1
AA = list("ACDEFGHIKLMNPQRSTVWY")

def generate_random_peptide(length):
    return "".join(
        random.choices(AA, k=length)
    )

non_amp_df = pd.DataFrame()

non_amp_df["SEQUENCE"] = [
    generate_random_peptide(length)
    for length in amp_df["length"]
]

non_amp_df["length"] = (
    non_amp_df["SEQUENCE"]
    .str.len()
)

non_amp_df["label"] = 0

Broj AMP peptida: 1672


## Formiranje klasa

Kako bi model mogao učiti razliku između dvije klase, potrebno je imati primjere antimikrobnih i neantimikrobnih peptida. Nakon pripreme podataka formira se konačni skup podataka koji sadrži obje klase i spreman je za daljnju obradu.

In [ ]:
AA = list("ACDEFGHIKLMNPQRSTVWY")

def generate_random_peptide(length):
    return "".join(
        random.choices(AA, k=length)
    )

non_amp_df = pd.DataFrame()

non_amp_df["SEQUENCE"] = [
    generate_random_peptide(length)
    for length in amp_df["length"]
]

non_amp_df["length"] = (
    non_amp_df["SEQUENCE"]
    .str.len()
)

non_amp_df["label"] = 0

print("Non-AMP peptidi:", len(non_amp_df))

Non-AMP peptidi: 1672


In [ ]:
final_dataset = pd.concat(
    [
        amp_df[["SEQUENCE", "label"]],
        non_amp_df[["SEQUENCE", "label"]]
    ],
    ignore_index=True
)

print("Ukupno uzoraka:", len(final_dataset))

final_dataset.head()

Ukupno uzoraka: 3344


,SEQUENCE,label
0,KVVVKWVVKVVK,1
1,LFIFFF,1
2,RVKRVWPLVIRTVIAGYNLYRAIKKK,1
3,RKRIHIGPGRAFYTT,1
4,GIWDTIKSMGKVFAGKILQNL,1


In [ ]:
final_dataset["label"].value_counts()

,count
label,
1,1672
0,1672


In [ ]:
final_dataset = final_dataset.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

final_dataset.head()

,SEQUENCE,label
0,CFQWQRNARKVR,1
1,GYFYHTEPTSHHRFWDFFISMFPYA,0
2,WLNALLHHGLNCAKGVL,1
3,RLWRIVVIRVAR,1
4,ILGTILGLLKSL,1


In [ ]:
final_dataset.to_csv(
    "amp_dataset.csv",
    index=False
)

print("Dataset spremljen.")

Dataset spremljen.


In [ ]:
print(final_dataset.shape)

final_dataset.head()

(3344, 2)


,SEQUENCE,label
0,CFQWQRNARKVR,1
1,GYFYHTEPTSHHRFWDFFISMFPYA,0
2,WLNALLHHGLNCAKGVL,1
3,RLWRIVVIRVAR,1
4,ILGTILGLLKSL,1


In [ ]:
import os

print(os.listdir())

for file in os.listdir():
    if file.endswith(".csv"):
        print(file)

print(os.listdir())

['.config', 'data', 'amp_dataset.csv', 'peptides.csv', 'sample_data']
amp_dataset.csv
peptides.csv
['.config', 'data', 'amp_dataset.csv', 'peptides.csv', 'sample_data']


## Zaključak

U ovom koraku pripremljen je skup podataka za daljnju analizu. Dobiven je uravnotežen skup podataka koji sadrži obje klase te je spremljen za ekstrakciju značajki u sljedećem dijelu projekta.
